# QTI Unified Test Drive

This notebook is a small, safe walkthrough of what this repo can do. It uses tiny synthetic settings by default, writes generated files under `data/notebook_test_drive`, and avoids requiring external MLP checkpoints or patient data unless you opt in near the end.


## 1. Local Setup

Run this first. It adds `src/` to `sys.path`, so you do not need to install the package just to explore it from the repo checkout.


In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "qti_unified").exists():
    REPO_ROOT = Path(r"C:/QTI_Unified")

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_ROOT = REPO_ROOT / "data" / "notebook_test_drive"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

print("Repo:", REPO_ROOT)
print("Data root:", DATA_ROOT)
print("pandas:", bool(importlib.util.find_spec("pandas")))
print("matplotlib:", bool(importlib.util.find_spec("matplotlib")))
print("nibabel:", bool(importlib.util.find_spec("nibabel")))
print("torch:", bool(importlib.util.find_spec("torch")))


## 2. See Available Scenarios

`qti_unified.patho` owns the patho scenario names and case definitions.


In [ ]:
from qti_unified.patho import CASE_NAMES, available_scenarios

scenarios = available_scenarios()
print(f"Scenarios: {len(scenarios)}")
for scenario in scenarios:
    print(f"- {scenario}: {len(CASE_NAMES[scenario])} case(s)")


## 3. Generate One Tiny Case

This creates one small patho case with stored GT, exact signal, cumulant signal, and two noisy realizations. It uses the built-in smoke-test protocol unless you pass an external XPS path.


In [ ]:
from qti_unified.config import PathoRunConfig
from qti_unified.pipeline import generate_patho_data, write_generation_manifest

config = PathoRunConfig(
    data_root=DATA_ROOT,
    n_tensors=16,
    n_realizations=2,
    snr=30.0,
    seed=3,
)

generated = generate_patho_data(config, scenarios=["needle_sphere_2"])
manifest = write_generation_manifest(generated, DATA_ROOT)

item = generated[0]
print("Generated cases:", len(generated))
print("Scenario/case:", item.case.scenario, item.case.case)
print("Manifest:", manifest)
print(json.dumps(item.paths, indent=2))


## 4. Inspect Stored GT

The workflow computes ground truth once during generation and stores it in `*_GT_params.json`. Prediction and plotting code read this file instead of recomputing GT.


In [ ]:
gt_path = Path(item.paths["gt_json"])
payload = json.loads(gt_path.read_text(encoding="utf-8"))

print("GT JSON:", gt_path)
print(json.dumps(payload["gt"], indent=2))
print("Metadata:")
print(json.dumps(payload["metadata"], indent=2))


## 5. Plot Exact, Cumulant, And Noisy Signals

This is a quick visual check of the generated signal vectors.


In [ ]:
import numpy as np

if importlib.util.find_spec("matplotlib"):
    import matplotlib.pyplot as plt

    x = np.arange(item.exact_signal.shape[0])
    plt.figure(figsize=(10, 4))
    plt.plot(x, item.exact_signal, label="exact", linewidth=2)
    plt.plot(x, item.cumexp_signal, label="cumexp", linestyle="--")
    if item.noisy_signals is not None:
        for i, noisy in enumerate(item.noisy_signals):
            plt.plot(x, noisy, alpha=0.55, label=f"noisy {i}")
    plt.xlabel("Measurement index")
    plt.ylabel("Signal")
    plt.title(f"{item.case.scenario} / {item.case.case}")
    plt.grid(True, ls=":", alpha=0.4)
    plt.legend(frameon=False)
    plt.show()
else:
    print("matplotlib is not installed, skipping plot.")


## 6. Discover Prediction Cases

This is what the MLP comparison step uses to find generated noisy signals and matching stored GT files.


In [ ]:
from qti_unified.pipeline import discover_prediction_cases

rows = discover_prediction_cases(DATA_ROOT, snr_folder="SNR30")
print("Prediction rows:", len(rows))
for row in rows[:5]:
    print(json.dumps(row, indent=2))


## 7. Optional: Generate All 14 Scenario GT Previews

Set `RUN_ALL_SCENARIOS = True` to generate a tiny GT-only version of every scenario and write simple preview plots. This is useful for a first repo checkup and does not need Torch or model checkpoints.


In [ ]:
RUN_ALL_SCENARIOS = False

if RUN_ALL_SCENARIOS:
    all_root = DATA_ROOT / "all_scenarios_gt_only"
    all_config = PathoRunConfig(
        data_root=all_root,
        n_tensors=16,
        n_realizations=0,
        snr=None,
        seed=42,
    )
    all_generated = generate_patho_data(all_config, scenarios=None)
    write_generation_manifest(all_generated, all_root)
    print("Generated cases:", len(all_generated))
    print("Root:", all_root)
else:
    print("Skipped. Set RUN_ALL_SCENARIOS = True to run this cell.")


In [ ]:
def write_gt_preview_plots(root: Path):
    if not importlib.util.find_spec("matplotlib"):
        print("matplotlib is not installed, skipping GT preview plots.")
        return []

    import matplotlib.pyplot as plt
    from qti_unified.plotting import parse_case_for_plotting, scenario_plot_name

    result_root = root / "Results_2_MLP_patho"
    out_dir = root / "runs" / "QTI_MLP_synthetic_compare_patho" / "gt_preview_plots"
    out_dir.mkdir(parents=True, exist_ok=True)
    plot_specs = [
        ("MD", "E[D_iso] [um2/ms]"),
        ("FA", "FA"),
        ("uFA", "uFA"),
        ("C_c", "C_c"),
        ("C_MD", "C_MD"),
    ]

    written = []
    for scenario_dir in sorted(p for p in result_root.iterdir() if p.is_dir()):
        scenario = scenario_dir.name
        rows = []
        for gt_json in sorted(scenario_dir.glob("*/*_GT_params.json")):
            payload = json.loads(gt_json.read_text(encoding="utf-8"))
            case = payload["case"]
            gt = payload["gt"]
            x, _, xlabel = parse_case_for_plotting(scenario, case)
            rows.append({
                "case": case,
                "x": x,
                "xlabel": xlabel,
                "MD": gt.get("MD_um2_per_ms"),
                "FA": gt.get("FA"),
                "uFA": gt.get("uFA"),
                "C_c": gt.get("C_c"),
                "C_MD": gt.get("C_MD"),
            })
        if not rows:
            continue

        rows.sort(key=lambda row: (row["x"], row["case"]))
        xs = np.asarray([row["x"] for row in rows], dtype=float)
        xlabel = rows[0]["xlabel"]

        fig, axes = plt.subplots(2, 3, figsize=(18, 9))
        axes = axes.ravel()
        for ax_i, (key, ylabel) in enumerate(plot_specs):
            ax = axes[ax_i]
            ys = np.asarray([np.nan if row[key] is None else float(row[key]) for row in rows], dtype=float)
            if len(rows) == 1:
                ax.plot(xs, ys, color="k", marker="*", linestyle="None", markersize=10)
                ax.set_xticks(xs, [rows[0]["case"]], rotation=20, ha="right")
            else:
                ax.plot(xs, ys, color="k", linestyle="--", marker="o", linewidth=1.5, markersize=4)
            ax.grid(True, ls=":", alpha=0.4)
            ax.set_xlabel(xlabel, fontsize=10)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_title(key)
        for ax in axes[len(plot_specs):]:
            ax.axis("off")
        fig.suptitle(f"{scenario.replace('_', ' ').title()} | GT preview", fontsize=14, y=1.02)
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        path = out_dir / scenario_plot_name(scenario).replace("_patho_cov.png", "_gt_preview.png")
        fig.savefig(path, dpi=200, bbox_inches="tight")
        plt.close(fig)
        written.append(path)

    return written

if RUN_ALL_SCENARIOS:
    preview_paths = write_gt_preview_plots(DATA_ROOT / "all_scenarios_gt_only")
    print("Wrote plots:", len(preview_paths))
    for path in preview_paths:
        print(path)
else:
    print("Skipped because RUN_ALL_SCENARIOS is False.")


## 8. Optional: Validate GT

This uses `pandas`, so it will skip cleanly if pandas is not installed.


In [ ]:
if importlib.util.find_spec("pandas"):
    from qti_unified.pipeline import validate_gt
    validation = validate_gt(DATA_ROOT)
    print(validation.head())
    print("Rows:", len(validation))
    print("Failures:", int((~validation["ok"].astype(bool)).sum()) if not validation.empty else 0)
else:
    print("pandas is not installed. Install with: python -m pip install -e .[dev]")


## 9. Optional: Full MLP Comparison

This needs `torch`, `pandas`, generated noisy signals, and external benchmark checkpoints. Set `RUN_MLP = True` and point `MODEL_ROOT` to your checkpoint folder.


In [ ]:
RUN_MLP = False
MODEL_ROOT = Path(r"C:/QTI_ML/BENCHMARK_ABSTRACT")
COV_FIT_ROOT = None  # Example: Path(r"C:/SynQTI-IR/data/Fit_Results/dtd_covariance_snr30_batch_sub_min_pp_patho")

if RUN_MLP:
    missing = [name for name in ["pandas", "torch"] if not importlib.util.find_spec(name)]
    if missing:
        raise RuntimeError(f"Missing required package(s): {missing}")
    if not MODEL_ROOT.exists():
        raise FileNotFoundError(f"MODEL_ROOT does not exist: {MODEL_ROOT}")

    from qti_unified.pipeline import compare_patho_predictions
    df = compare_patho_predictions(
        data_root=DATA_ROOT,
        model_root=MODEL_ROOT,
        cov_fit_root=COV_FIT_ROOT,
        snr_folder="SNR30",
        device="cpu",
    )
    print(df.head())
else:
    print("Skipped. Set RUN_MLP = True after installing dependencies and setting MODEL_ROOT.")


## 10. Optional: In-Vivo Patho Winner Plots

This recreates the `invivo_compare_patho/*_winner_all4.png` figures from the repo helper ported from `C:/SynQTI-IR/SynDTDs_MLP_patho.ipynb` cell 52. It needs external patient signal, mask, XPS files, and generated patho exact signals.


In [ ]:
RUN_INVIVO_WINNERS = False

INVIVO_RESULTS_ROOT = Path(r"C:/SynQTI-IR/data/Results_2_MLP_patho")
BRAIN_SIGNAL = Path(r"C:/SynQTI-IR/P14/og_NII_dn_db_dg_tp_mc_b0_avg.nii.gz")
BRAIN_XPS = Path(r"C:/SynQTI-IR/P14/xps_sub_min_pp.mat")
BRAIN_FULL_XPS = Path(r"C:/SynQTI-IR/P14/xps_full_mc.mat")
BRAIN_MASK = Path(r"C:/SynQTI-IR/P14/manual_mask.nii.gz")
INVIVO_OUT = DATA_ROOT / "invivo_compare_patho"

if RUN_INVIVO_WINNERS:
    from qti_unified.invivo import export_invivo_patho_winner_plots
    summary = export_invivo_patho_winner_plots(
        results_root=INVIVO_RESULTS_ROOT,
        brain_signal_path=BRAIN_SIGNAL,
        brain_xps_path=BRAIN_XPS,
        brain_full_xps_path=BRAIN_FULL_XPS,
        brain_mask_path=BRAIN_MASK,
        output_dir=INVIVO_OUT,
        z_idx=10,
    )
    print(summary)
    print("Output:", INVIVO_OUT)
else:
    print("Skipped. Set RUN_INVIVO_WINNERS = True when the external P14 files are available.")


## 11. Optional: Patient Comparison

This ranks generated exact patho signals against an external patient signal. It never copies patient data into the repo.


In [ ]:
RUN_PATIENT_COMPARE = False
PATIENT_SIGNAL = Path(r"D:/patients/P01/signal.nii.gz")

if RUN_PATIENT_COMPARE:
    if not PATIENT_SIGNAL.exists():
        raise FileNotFoundError(PATIENT_SIGNAL)
    from qti_unified.pipeline import compare_patient_target
    patient_df = compare_patient_target(
        patient_signal_path=PATIENT_SIGNAL,
        data_root=DATA_ROOT,
        target="Brain mean",
    )
    print(patient_df.head(10))
else:
    print("Skipped. Set RUN_PATIENT_COMPARE = True and PATIENT_SIGNAL to a real external file.")


# Invivo compare

In [ ]:
from pathlib import Path
from qti_unified.invivo import export_invivo_patho_winner_plots

summary = export_invivo_patho_winner_plots(
    results_root=Path(r"C:\SynQTI-IR\data\Results_2_MLP_patho"),
    brain_signal_path=Path(r"C:\SynQTI-IR\P14\og_NII_dn_db_dg_tp_mc_b0_avg.nii.gz"),
    brain_xps_path=Path(r"C:\SynQTI-IR\P14\xps_sub_min_pp.mat"),
    brain_full_xps_path=Path(r"C:\SynQTI-IR\P14\xps_full_mc.mat"),
    brain_mask_path=Path(r"C:\SynQTI-IR\P14\manual_mask.nii.gz"),
    output_dir=Path(r"C:\QTI_Unified\data\invivo_compare_patho"),
    z_idx=10,
)
